In [1]:
# !pip install selenium beautifulsoup4 webdriver-manager

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceException, TimeoutException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import time
from datetime import datetime
import re

In [103]:
# detail test
options = Options()
options.page_load_strategy = 'eager'

driver = webdriver.Chrome(options=options) 
detail_href = "https://place.map.kakao.com/1086385576"
driver.get(detail_href)


In [109]:
html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')

# 한줄평
tag = soup.select_one('.desc_boxintro')
simple_desc = tag.get_text(strip=True).replace("\n", "") if tag else ""

print("한줄평은 다음과 같다: ", simple_desc)

# 태그
def extract_tags(soup):
    tags = []
    for el in soup.find_all(['span', 'a']):
        text = el.get_text(strip=True)
        if text.startswith('#') and len(text) < 20:
            tags.append(text)
    return list(dict.fromkeys(tags))  # 중복 제거

tags = extract_tags(soup)
print("태그는 다음과 같다: ", tags)

# 영업시간
# 'btn_fold2' => 눌러서 열고
# fold_detail 읽고 line_fold 하나씩 읽기
# 거기서 tit_fold 읽고 txt_detail 로 읽기 tit_fold 는 날짜만 읽기

# 더보기 버튼 누르기
def click_opening_hour_more(driver):
    try:
        btn = WebDriverWait(driver, 3).until(
            EC.element_to_be_clickable((By.CLASS_NAME, "btn_fold2"))
        )
        driver.execute_script("arguments[0].click();", btn)
        time.sleep(1)
        return True
    except:
        return False
    
def parse_opening_hours(driver):
    soup = BeautifulSoup(driver.page_source, "html.parser")

    result = []

    fold_area = soup.select_one(".fold_detail")
    if not fold_area:
        return result  # 상세 페이지 없음

    lines = fold_area.select(".line_fold")

    for line in lines:
        tit = line.select_one(".tit_fold")
        date_text = tit.get_text(strip=True) if tit else ""

        date_only = re.findall(r"[월화수목금토일]", date_text)
        date_only = date_only[0] if date_only else date_text

        # 영업시간
        txt = line.select_one(".txt_detail")
        time_text = txt.get_text(" ", strip=True) if txt else ""

        result.append({
            "day": date_only,
            "hours": time_text
        })

    return result

# 더보기 버튼 클릭
click_opening_hour_more(driver)

opening_hours = parse_opening_hours(driver)
if opening_hours:
    # info['영업시간'] = opening_hours
    for oh in opening_hours:
        print(f"{oh['day']}: {oh['hours']}")
else:
    print("영업시간 상세 정보 없음")


# 가격 정보 가져오기
price_list = soup.select('.section_product .list_goods > li')
print(f"총 {len(price_list)}개의 가격 정보 수집 완료")
prices = {}
for _, room in enumerate(price_list):
    room_type = extract_info(room, '.tit_item')
    price = extract_info(room, '.desc_item').replace("~", "")
    if room_type is None or price is None:
        continue
    prices[room_type] = price
print("가격: ", prices)
    
# 리뷰 갯수
number_of_reviews = soup.select('.info_num')[0].get_text(strip=True) if soup.select('.info_num') else None
print(number_of_reviews)

# 카테고리 모텔 호텔 종류
raw_text = soup.select('.info_cate')[0]
category = raw_text.get_text(strip=True) if raw_text else None
category = category.replace("장소 카테고리", "").replace("\"", "").strip() if category else None

print("카테고리는 다음과 같다: ", category)

# 리뷰
reviews = []
reviews = get_recent_reviews(driver)
# 리뷰 없으면 해당 숙소 패스
if reviews is None or len(reviews) <= 3:
    print("there's no review")
    driver.close()
    time.sleep(2)
print("리뷰: ", reviews)

time.sleep(3)


InvalidSessionIdException: Message: invalid session id; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
0   chromedriver                        0x0000000102863dfc cxxbridge1$str$ptr + 3031016
1   chromedriver                        0x000000010285bcb8 cxxbridge1$str$ptr + 2997924
2   chromedriver                        0x00000001023569f4 _RNvCsgXDX2mvAJAg_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 73780
3   chromedriver                        0x00000001023915f8 _RNvCsgXDX2mvAJAg_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 314424
4   chromedriver                        0x00000001023b9520 _RNvCsgXDX2mvAJAg_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 478048
5   chromedriver                        0x00000001023b8954 _RNvCsgXDX2mvAJAg_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 475028
6   chromedriver                        0x0000000102325784 chromedriver + 10

In [8]:
#  리뷰 가져오기 제일 최신 순으로 3개

# 최신순 정렬
def click_latest_sort(driver):
    buttons = driver.find_elements(By.CSS_SELECTOR, "a.link_sort")

    for btn in buttons:
        text = driver.execute_script(
            "return arguments[0].textContent;", btn
        ).strip()

        if "최신" in text:
            if btn.find_elements(By.CSS_SELECTOR, ".ico_chk"):
                return True

            driver.execute_script(
                "arguments[0].scrollIntoView({block:'center'});", btn
            )
            time.sleep(0.3)

            driver.execute_script("arguments[0].click();", btn)
            time.sleep(2)

            return True

    return False
        
# 더보기 버튼 누르기
def load_reviews_with_more(driver):
    try:
        more_buttons = driver.find_elements(By.CSS_SELECTOR, ".btn_more")
        for btn in more_buttons:
            if btn.is_displayed():
                driver.execute_script("arguments[0].click();", btn)
                time.sleep(0.8)
    except:
        pass 
    

def get_recent_reviews(driver, year=2025, max_count=2):
    driver.execute_script("location.hash = '#review'")
    driver.switch_to.window(driver.window_handles[-1])
    
    print("리뷰 페이지로 이동 중...", driver.current_url)
    
    # 로딩 대기
    time.sleep(8)

    click_latest_sort(driver)
    load_reviews_with_more(driver)
    
    time.sleep(20)

    
    soup = BeautifulSoup(driver.page_source, "html.parser")
    review_list = soup.select('.list_review .inner_review') # list 가져오기

    print(f"총 {len(review_list)}개의 리뷰 정보 수집 완료")

    reviews = []
    review_count = 0
    for r in review_list:
        date = r.select('.txt_date')[0].text if r.select('.txt_date') else None
        if date is None:
            continue
        review_date = datetime.strptime(date, "%Y.%m.%d.")


        if review_date.year == year and review_count < max_count:
            describe = r.select('.desc_review')[0].text if r.select('.desc_review') else None 
            if describe is not None:
                review_count += 1
                
                # simple cleaning
                describe = describe.replace('접기', '')
                describe = describe.replace('\n', ' ')
                describe = re.sub(r'[^가-힣a-zA-Z0-9\s.,!?]', '', describe)

                reviews.append(describe)
            else:
                continue
            
        elif review_date.year > year:
            continue
        elif review_count >= max_count:
            break
        else:
            break
    # tab close
    time.sleep(2)
    return reviews

In [9]:
def extract_tags(soup):
    tags = []
    for el in soup.find_all(['span', 'a']):
        text = el.get_text(strip=True)
        if text.startswith('#') and len(text) < 20:
            tags.append(text)
    return list(dict.fromkeys(tags))  # 중복 제거

def extract_info(soup, tag_name):
    tag = soup.select_one(tag_name)
    text = tag.get_text(strip=True).replace("\n", "") if tag else None
    return text

# 영업 시간 더보기 버튼 누르기
def click_opening_hour_more(driver):
    try:
        btn = WebDriverWait(driver, 3).until(
            EC.element_to_be_clickable((By.CLASS_NAME, "btn_fold2"))
        )
        driver.execute_script("arguments[0].click();", btn)
        time.sleep(1)
        return True
    except:
        return False
    
# 영업 시간
def parse_opening_hours(driver):
    soup = BeautifulSoup(driver.page_source, "html.parser")

    result = []

    fold_area = soup.select_one(".fold_detail")
    if not fold_area:
        return result  # 상세 페이지 없음

    lines = fold_area.select(".line_fold")

    for line in lines:
        tit = line.select_one(".tit_fold")
        date_text = tit.get_text(strip=True) if tit else ""

        date_only = re.findall(r"[월화수목금토일]", date_text)
        date_only = date_only[0] if date_only else date_text

        # 영업시간
        txt = line.select_one(".txt_detail")
        time_text = txt.get_text(" ", strip=True) if txt else ""

        result.append({
            "day": date_only,
            "hours": time_text
        })

    return result

In [10]:
# 디테일 전체 함수
def extract_details(detail_href):
    info = {}
    
    # 메인 페이지
    main_window = driver.current_window_handle
    time.sleep(5)
    
    driver.execute_script("window.open(arguments[0]);", detail_href)
    driver.switch_to.window(driver.window_handles[-1])
    
    print("상세 페이지로 이동 중...", driver.current_url)
    
    # 로딩 대기
    time.sleep(15)
    
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')

    # 한줄설명
    simple_desc = extract_info(soup, '.desc_boxintro')
    print("한줄설명: ", simple_desc)
    info['한줄설명'] = simple_desc
    
    # 리뷰 갯수
    number_of_reviews = extract_info(soup, '.info_num')
    print("리뷰 개수: ", number_of_reviews)
    info['리뷰 개수'] = number_of_reviews
    
    # 태그
    tags = extract_tags(soup)
    print("태그: ", tags)
    info['태그'] = tags
    
    # 영업시간
    # 더보기 버튼 클릭
    click_opening_hour_more(driver)

    opening_hours = parse_opening_hours(driver)
    if opening_hours:
        for oh in opening_hours:
            print(f"{oh['day']}: {oh['hours']}")
    else:
        print("영업시간 상세 정보 없음")
    info['영업시간'] = opening_hours # 없어도 넣고 있어도 넣음


    # 가격 정보 가져오기
    price_list = soup.select('.section_product .list_goods > li')
    print(f"총 {len(price_list)}개의 가격 정보 수집 완료")
    prices = {}
    for _, room in enumerate(price_list):
        room_type = extract_info(room, '.tit_item')
        price = extract_info(room, '.desc_item')
        if room_type is None or price is None:
            continue
        prices[room_type] = price
    print("가격: ", prices)
        
    # 리뷰
    max_review_count = 2
    reviews = []
    reviews = get_recent_reviews(driver, max_count=max_review_count)
    # 리뷰 없으면 해당 숙소 패스
    if reviews is None or len(reviews) < max_review_count:
        driver.close()
        driver.switch_to.window(main_window)
        time.sleep(2)
        return None
    print("리뷰: ", reviews)
    info['리뷰'] = reviews

    # 다시 검색 탭으로 전환
    driver.close()
    driver.switch_to.window(main_window)
    time.sleep(2)
    return info


In [11]:
# for testing 1 page
# 2 methods needed

def crawling(accommodation_list):
    informations = []
    for _, accommodation in enumerate(accommodation_list):
            info = {}
        # 장소명
            name = extract_info(accommodation, '.head_item > .tit_name > .link_name') 
            print("장소명:", name)
            info['장소명'] = name
        # 장소명 없으면 pass
            if name is None: 
                    continue
        
        # 평점
            score = extract_info(accommodation, '.rating > .score > em')
            print("평점:", score)
            info['평점'] = score
        # 평점 없으면 pass
            if score is None:
                    continue 
        
        # 주소
            addr = extract_info(accommodation, '.addr > p')
            print("주소:", addr)
            info['주소'] = addr
        # 주소가 없거나, 부산으로 시작하지 않으면 pass
            if addr is None or not addr.startswith("부산"):
                    continue

        # 구분
            info['구분'] = "체험"
        
        # 카테고리
            category = extract_info(accommodation, '.subcategory')
            print("카테고리:", category)
            info['카테고리'] = category
        
        # 연락처
            phone_number = extract_info(accommodation, '.phone')
            print("연락처:", phone_number)
            info['연락처'] = phone_number

            a_tag = accommodation.select_one('.moreview')
            detail_href = a_tag.get('href') if a_tag else None
            details = extract_details(detail_href)
            if details is None:
                    continue
        
            info.update(details)
            info['출처'] = "카카오맵(2025)"
            print()
            print("--------------------------------------------------------------------------------------------------------")
            print("all the info: ", info)
            print("--------------------------------------------------------------------------------------------------------")
            informations.append(info)
            
    return informations

        

In [12]:
# kakao map crawling
options = Options()
options.page_load_strategy = 'eager'

driver = webdriver.Chrome(options=options) 
url = "https://map.kakao.com/"
driver.get(url)

searchloc = '부산 해수온천'

search_area = driver.find_element(
    By.CSS_SELECTOR,
    "input[type='text']"
)
search_area.send_keys(searchloc)  # 검색어 전달
search_area.send_keys(Keys.ENTER)

time.sleep(5)

driver.find_element(By.XPATH, r'//*[@id="info.main.options"]/li[2]/a').send_keys(Keys.ENTER) # 장소 탭 이동

In [ ]:
# next_btn = driver.find_element(By.XPATH, r'//* [@id="info.search.page.next"]')
# next_btn.send_keys(Keys.ENTER)

# xpath = f'/html/body/div[5]/div[2]/div[1]/div[7]/div[6]/div/a[2]'
# driver.find_element(By.XPATH, xpath).send_keys(Keys.ENTER)

In [13]:
import pandas as pd
import os
# 저장할 파일명
filename = "busan_activity_result.csv"

page = 1   # 현재 전체 페이지 카운트
page2 = 1  # 1~5번 버튼 중 몇 번째인지

for i in range(1, 11):
    try:
        print(f"\n=== [ {page} 페이지 수집 시작 ] ===")
        
        btn_index = page2 # 현재 그룹(1~5) 내에서의 인덱스

        xpath = f'/html/body/div[5]/div[2]/div[1]/div[7]/div[6]/div/a[{btn_index}]'
        
        try:
            driver.find_element(By.XPATH, xpath).send_keys(Keys.ENTER)
        except:
            pass
            
        time.sleep(2) 
        
        html = driver.page_source
        soup = BeautifulSoup(html, 'html.parser')

        accommodation_list = soup.select('.placelist > .PlaceItem')
        print(f"찾은 업체 개수: {len(accommodation_list)}개")
        
        if len(accommodation_list) == 0:
            print("데이터가 없습니다. (혹시 로딩 에러?)")
            continue

        # 크롤링 함수 실행
        informations = crawling(accommodation_list)
        print(f"수집 완료: {len(informations)} 건")

        df = pd.DataFrame(informations)
        
        # 파일이 없으면(첫 페이지면) -> 헤더 포함해서 새로 만들기 ('w' 모드)
        if not os.path.exists(filename):
            df.to_csv(filename, index=False, encoding='utf-8-sig', mode='w')
        # 파일이 있으면(2페이지부터) -> 헤더 빼고 이어 붙이기 ('a' 모드)
        else:
            df.to_csv(filename, index=False, encoding='utf-8-sig', mode='a', header=False)
            
        print(f"-> '{filename}'에 저장 완료!")
        
        page += 1
        page2 += 1
        
        # page2가 5를 넘어가면 (즉, 6번째가 되면) -> '다음' 화살표 클릭하고 page2 초기화
        if page2 > 5:
            page2 = 1
            print(">> '다음' 버튼 클릭 (페이지 그룹 이동)")
            next_btn = driver.find_element(By.XPATH, r'//* [@id="info.search.page.next"]')
            
            # 다음 버튼이 비활성화(disabled) 상태면 종료
            if next_btn.get_attribute("disabled"):
                print("더 이상 다음 페이지가 없습니다. 종료합니다.")
                break
                
            next_btn.send_keys(Keys.ENTER)
            time.sleep(2) # 다음 묶음 로딩 대기

    except Exception as e:
        print(f"에러 발생 (중단): {e}")
        break
        
print('\n크롤링 전체 완료!')


=== [ 1 페이지 수집 시작 ] ===
찾은 업체 개수: 15개
장소명: 금샘해수온천
평점: 4.4
주소: 부산 강서구 명지국제8로 237 금샘빌딩 5층
카테고리: 온천
연락처: 0507-1361-8867
상세 페이지로 이동 중... https://place.map.kakao.com/2125242092
한줄설명:  금샘빌딩 주차장(B1~B3)만차시 명지국제타워 주차장에 주차 가능합니다.외부음료음식반입X
리뷰 개수:  30개
태그:  ['#광천온천', '#명지사우나', '#명지온천', '#부산온천', '#황산염온천', '#해수온천']
수: 06:00 ~ 22:00
목: 06:00 ~ 22:00
금: 06:00 ~ 22:00
토: 06:00 ~ 22:00
일: 06:00 ~ 22:00
월: 휴무일
화: 06:00 ~ 22:00
총 2개의 가격 정보 수집 완료
가격:  {'온천(일반)': '10,000원', '온천(소인)': '6,000원'}
리뷰 페이지로 이동 중... https://place.map.kakao.com/2125242092#review
총 20개의 리뷰 정보 수집 완료
리뷰:  ['울신랑이 가보고 물이 넘 좋다고 해서 저혼자 가봤어요 . 시설도 깨끗하고 특이하게 탕이 탈으실과 다른층에 있어 색다르고 좋았어요 배치된 시설물들도 맘에 들었어요 탕 온도가 좀 미지근해서 놀라웠지만 담글수록 알맞은 온도인게 느껴졌어요 땀이 아주 잘나오고 때가 장난 아니게 잘 나온답니다 어제 다녀왔는데 오늘 아침에 얼굴이 반딱거려 넘 좋아요  ', '물도 좋고 친절하고 다좋습니다. 애들을 데리고 오면서 물총 물안경 이딴것좀 들고오지마라. 여기가 수영장이가? 조용히 목욕좀 하자 제발좀 몰상식한 인간들아 목욕탕 갈때마다 분노가 치솟네  ']

--------------------------------------------------------------------------------------------------------
all the info:  {'장소명': '금샘